# Guided Ad Hoc Analysis — OrbitMart

Diagnose a week-over-week checkout-completion decline within a four-hour analysis contract. The company, events, and data are synthetic.

**Decision boundary:** localizing a decline does not prove its cause. This notebook separates observed facts, diagnostic evidence, supported hypotheses, and unknowns.


## 1. Load the reproducible workflow

The helper locates the module whether the notebook runs from its own folder or the repository root.


In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]
module_root = next(root / "04_ad_hoc_analysis" for root in candidate_roots if (root / "04_ad_hoc_analysis").exists())

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

analysis = load_module("adhoc_analysis", module_root / "src" / "diagnose_kpi_change.py")
generator = load_module("adhoc_generator", module_root / "src" / "generate_synthetic_data.py")

full_path = module_root / "data" / "raw" / "orbitmart_checkout_diagnostic_full.csv"
raw = pd.read_csv(full_path) if full_path.exists() else generator.generate_raw_data()
raw.shape


## 2. Validate before explaining

Preserve the raw report. Normalize dimensions, label unknown channels, deduplicate the declared grain, and quarantine impossible funnels.


In [ ]:
quality = analysis.quality_report(raw)
clean = analysis.clean_data(raw)
quality, clean.shape


## 3. Confirm the headline on matched periods

Compute rates from summed numerators and denominators. The agreed materiality threshold is 0.50 percentage points.


In [ ]:
summary = analysis.period_summary(clean)
summary.loc[[
    "checkout_starts", "orders_completed", "checkout_completion_rate",
    "revenue_usd", "support_contacts"
]].round(4)


## 4. Use the KPI tree to choose the next branch

Attempt rate × approval rate × post-approval completion rate equals checkout completion. Approval is the only materially changing stage.


In [ ]:
tree = analysis.kpi_tree(clean)
tree.assign(
    prior=lambda frame: frame["prior"].map(lambda value: f"{value:.2%}"),
    current=lambda frame: frame["current"].map(lambda value: f"{value:.2%}"),
    absolute_change=lambda frame: frame["absolute_change"].map(lambda value: f"{100 * value:+.2f} pp"),
)


## 5. Run bounded segment diagnostics

The pre-specified dimensions use a 300-start minimum, a two-percentage-point materiality rule, and Benjamini–Hochberg correction within each family.


In [ ]:
dimensions = ["country", "platform", "app_version", "payment_method", "acquisition_channel"]
diagnostics = {dimension: analysis.segment_diagnostics(clean, dimension) for dimension in dimensions}
diagnostics["app_version"][[
    "prior_starts", "current_starts", "prior_rate", "current_rate",
    "change_pp", "within_effect_pp", "mix_effect_pp", "q_value", "material_decline"
]].round(4)


In [ ]:
diagnostics["payment_method"][[
    "prior_starts", "current_starts", "change_pp",
    "within_effect_pp", "mix_effect_pp", "q_value", "material_decline"
]].round(4)


## 6. Quantify impact without overstating causality

Apply the prior completion rate to the current denominator and value the estimated order gap at prior average order value.


In [ ]:
impact = analysis.impact_estimate(clean)
impact


## 7. Decision readout

**Facts:** checkout starts rose 0.9%, completion fell 0.97 pp, payment approval fell 0.96 pp, and support contacts rose 47.2%.

**Supported hypothesis:** the decline concentrates in Android 8.4 digital-wallet traffic. Within-segment effects dominate traffic-mix effects.

**Unknown:** aggregate data cannot distinguish whether the Android release, wallet processor, or their interaction caused the issue.

**Action:** pause Android 8.4 expansion in Mexico and Brazil, inspect wallet error signatures, apply a reversible routing or rollback mitigation if confirmed, and monitor approval recovery against Android 8.3 and non-wallet cohorts.

**Stop:** the urgent decision is supported. Assign request-level log analysis and mitigation measurement as follow-up work rather than continuing unbounded slicing.
